### Detecting Bar or Stripe Pattern in Pixel Grid with SuperNeuroMAT

In the following tutorial, we will create a spiking neural network that, given 9 binary inputs (representing pixels on a 3x3 screen), will return whether they form a vertical (bar) or horizontal (stripe) line.

![bars and stripes example images](img/bars_and_stripes_images/GridExamples.png)

For our variation of this problem, the algorithm will only return a "bar" or "stripe" output when there is a single column or row of enabled pixels. Therefore, there must be 3 enabled pixels for the algorithm to return either output.

In [ ]:
# If you haven't yet installed SuperNeuroMat, copy the following and run it in your terminal: 
# pip install superneuromat

In [ ]:
# Now let's import the SuperNeuroMAT library and name our algorithm:
import superneuromat as snm
BSA = snm.SNN()
# BSA stands for bar/stripe algorithm

We will make three layers of neurons: the input layer (where we add spikes representing enabled pixels), the hidden layer (where the logic occurs), and the output layer (signaling bar or stripe).

### Input Layer
The data will be inputted as a list of nine integers, each of which can be a 0 (representing a blank pixel) or a 1 (representing an enabled pixel). For the input layer we will create 9 neurons with default parameters, each corresponding to a pixel. This is the layer we will manually add spikes to.

In [ ]:
# First we make an empty list for inputs:
inputs = []

# The .create_neuron() method returns each neurons id, so the following will add ids to the list:
for i in range(9):
    id = BSA.create_neuron(threshold=0)
    inputs.append(id)

print(BSA.neurons)

![input layer image](img/bars_and_stripes_images/InputLayer.png) 

### Hidden Layer
We know that there are 6 configurations that should return a true output, consisting of each row (3) and column (3) being enabled. 

For our logic layer we can make 6 neurons that correspond to these configurations and spike when they occur. Since they must each receive 3 spikes from the input layer to return true, we'll set the threshold to 2:

In [ ]:
hidden = []

for i in range(6):
    id = BSA.create_neuron(threshold=2)
    hidden.append(id)
    
print(BSA.neurons)

![hidden layer image](img/bars_and_stripes_images/HiddenLayer.png) 

<small>*diagram depicts neurons' system ids for reference on spike train<br>
(ex. Neuron 9 in the diagram and spike train is the same as hidden[0])* </small>

### Output Layer

For our final layer we will create two output neurons, one indicating a bar and the other indicating a stripe. These will each have the default threshold of 0 because if any of the hidden neurons spike, their corresponding outputs should also spike.

In [ ]:
outputs = []
for i in range(2):
    id = BSA.create_neuron(threshold=0)
    outputs.append(id)

print(BSA.neurons)

![output layer image](img/bars_and_stripes_images/OutputLayer.png) 

### Cancel Neuron

To prevent the output neurons spiking when there are extra enabled pixels (more than 3), we must create one more neuron that spikes when this occurs. This neuron will connect to the output neurons through a negatively weighted synapse, negating the hidden layers' signals.

In [ ]:
Cancel = BSA.create_neuron(threshold=3)

All system neurons have been created:

![cancel neuron image](img/bars_and_stripes_images/CancelNeuron.png) 

### Creating Synapses

Now that we've created all of our neurons we can start making synapses. The hidden layer neurons are the "detectors", so most of the logic will be done by connecting the correct input layer neurons to their corresponding hidden layer neuron.

![detection grid image](img/bars_and_stripes_images/DetectionGrid.png) 

<small> *circles (hidden indices) 0-5 correspond to neuron ids 9-14 in the previous diagram and spike train* </small>

The diagram above shows which input neurons indices (pixels) trigger each hidden layer neuron index (circles). 

Vertical arrows represent bar detection and horizontal arrows represent stripe detection.

In [ ]:
# Stripe Detection: 
# Each target stripe neuron's index equals input floor divided by 3 because floor division removes decimals.
# (ex. 2//3 = 0, 3//3 = 1, 7//3 = 2), this matches the horizontal logic of our diagram above.
for i in inputs:
    BSA.create_synapse(i, hidden[inputs.index(i)//3])
print(BSA.synapses)
# The terminal also matches our full system diagram because hidden[0] has a system id of 9 (and so on).

![stripe detection image](img/bars_and_stripes_images/StripeDetection.png) 

In [ ]:
# Bar Detection:
# We will use modulo(%) by 3, because inputs/3 has a remainder of 0, 1, or 2, sorting them correctly into different hidden neurons
# We add 3 to the result to call correct hidden index
# [ex. (3%0)+3 = 3, (7%3)+3 = 4, (8%3)+3 = 5], this matches the vertical logic of our diagram above.
for i in inputs:
    BSA.create_synapse(i, hidden[(inputs.index(i)%3)+3])
print(BSA.synapses)
# See synapses 9-17 in the terminal:

![bar detection image](img/bars_and_stripes_images/BarDetection.png) 

Next, the hidden layer neurons pass their spike to an output neuron. Since hidden[0, 1, 2] detect a stripe and hidden[3, 4, 5] detect a bar, we can use integer division again to link them to their corresponding output neuron.

In [ ]:
# output[0] is our stripe neuron, output[1] is our bar neuron
# We use // 3 to get output[0] from hidden[0, 1, 2] and output[1] from hidden[3, 4, 5] 
for i in hidden:
    BSA.create_synapse(i, outputs[hidden.index(i)//3])

Our detection system is now set up:

![detection system diagram](img/bars_and_stripes_images/Detection.png) 

Now we just need to avoid false positives. 

The cancel neuron, which accepts spikes from all inputs, will spike if more than 3 pixels are enabled (we made it threshold=3 during creation).

In [ ]:
for i in inputs:
    BSA.create_synapse(i, Cancel)


By weighing the synapses from cancel to output negatively, we can negate either from spiking when two many pixels are enabled. 

In [ ]:
for i in outputs:
    BSA.create_synapse(Cancel, i, weight=-3)
print(BSA.synapses)
# see synapses (24-34)

We have finished the system. Here is our completed diagram: 

![completed diagram](img/bars_and_stripes_images/CompletedDiagram.png) 

### Adding Spikes and Inferencing

Lastly, there are a few lines we have to add to test our network and generalize it for varying frame sizes.

In [ ]:
spikes = list("100100100")
# This creates an easily-manipulable list of numeric strings
# The BSA.add_spike() requires three parameters: (time, neuron, spike value)
for i in inputs:
    BSA.add_spike(0, i, spikes[inputs.index(i)])
# This loop adds a spike to each neuron with its corresponding value from the spikes list  
print(BSA)

In [ ]:
# This network will take 3 time steps (0, 1, and 2) to return our output. 
BSA.simulate(3)
# Spikes enter at t=0, travel to the hidden layer at t=1, then register on the output layer at t=2 
# The negating spikes travel to the cancel neuron at t=1 and also negate the output layer at t=2 
# Each neuron is spiked for one time step because leak is infinite, so our result is shown solely at t=2
print(BSA)

First let's rearrange our code by putting variable definitions at the top.

In [ ]:
import superneuromat as snm
# Making stripe bar algorithm (BSA):
BSA = snm.SNN()

#Lists:
inputs = []
hidden = []
outputs = []
spikes = list("100100100")

# ******Neurons******
# Create input layer:
for i in range(9):
    id = BSA.create_neuron(threshold=0)
    inputs.append(id)

# Create hidden layer:
for i in range(6):
    id = BSA.create_neuron(threshold=2)
    hidden.append(id)

# Create output layer:
for i in range(2):
    id = BSA.create_neuron(threshold=0)
    outputs.append(id)

# Create Cancel neuron:
Cancel = BSA.create_neuron(threshold=3)

# ******Synapses******
# stripe detection:
for i in inputs:
    BSA.create_synapse(i, hidden[inputs.index(i)//3])
print(BSA.synapses)
# bar detection:
for i in inputs:
    BSA.create_synapse(i, hidden[(inputs.index(i)%3)+3])
print(BSA.synapses)

# hidden to outputs:
for i in hidden:
    BSA.create_synapse(i, outputs[hidden.index(i)//3])
# inputs to Cancel:
for i in inputs:
    BSA.create_synapse(i, Cancel)
# Cancel to outputs:
for i in outputs:
    BSA.create_synapse(Cancel, i, weight=-3)

# ******Spikes******
for i in inputs:
    BSA.add_spike(0, i, spikes[inputs.index(i)])

BSA.simulate(3)
print(BSA)

If we want to generalize this algorithm to accept all square frames, we can represent each side as being n pixels long:

In [ ]:
# Our layers are now: n^2 input neurons, 2n hidden neurons, and still 2 output neurons

# We replace the loop ranges with these values, 
# set Cancel threshold and Cancel-output synapse weight to n, 
# and make bar and stripe detection algorithms use n instead of 3: 

# (essentially: fix loop ranges, replace all 3's with n but keep simulation time constant)   

import superneuromat as snm
# Making stripe bar algorithm (BSA):
BSA = snm.SNN()
#Layers:
inputs = []
hidden = []
outputs = []
spikes = list("1000100010001000")
#Pixel Size:
n=4
# ******Neurons******
# Create input layer:
for i in range(n**2):
    id = BSA.create_neuron(threshold=0)
    inputs.append(id)

# Create hidden layer:
for i in range(2*n):
    id = BSA.create_neuron(threshold=2)
    hidden.append(id)

# Create output layer:
for i in range(2):
    id = BSA.create_neuron(threshold=0)
    outputs.append(id)

# Create Cancel neuron:
Cancel = BSA.create_neuron(threshold=n)

# ******Synapses******
# stripe detection:
for i in inputs:
    BSA.create_synapse(i, hidden[inputs.index(i)//n])
print(BSA.synapses)
# bar detection:
for i in inputs:
    BSA.create_synapse(i, hidden[(inputs.index(i)%n)+n])
print(BSA.synapses)

# hidden to outputs:
for i in hidden:
    BSA.create_synapse(i, outputs[hidden.index(i)//n])
# inputs to Cancel:
for i in inputs:
    BSA.create_synapse(i, Cancel)
# Cancel to outputs:
for i in outputs:
    BSA.create_synapse(Cancel, i, weight=-n)

# ******Spikes******
for i in inputs:
    BSA.add_spike(0, i, spikes[inputs.index(i)])

BSA.simulate(3)
print(BSA)

Congratulations, you have completed this tutorial!